# Demo 1 --- Same task, two systems

Three customer messages, two systems handling them. The default is the ordinary pipeline: classify the message, retrieve nearby policy text, let a model draft a reply. The governed system is the capstone harness: a fixed tool workflow behind a gate stack, with an escalation path and an audit log. Both are run below on the same three cases; the output is real.

In [ ]:
import json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

root = next((c for c in (Path('.'), Path('..'), Path('../..'), Path('../../code'), Path('../code'))
             if (c / 'data' / 'eval_cases' / 'cases.json').exists()), Path('.'))
cases = {c['id']: c for c in json.loads((root / 'data' / 'eval_cases' / 'cases.json').read_text())}
demo_ids = ['case-001', 'case-016', 'case-011']
for cid in demo_ids:
    print(cid, '->', cases[cid]['message'])

## The default: it always drafts

The baseline is the classifier plus the `DenseRagRetriever` --- textbook chunk-and-pray retrieval that embeds the policy document, pulls the nearest chunks and lets the model answer from them. It has no gate, no grounding check and no abstention: `decision` is always `answer` and `verified` is always `False`. Every case gets a confident reply, including the one carrying an SSN.

In [ ]:
from agentlab.models.complaint_classifier import get_default_classifier
from agentlab.capstone.dense_rag import DenseRagRetriever

clf = get_default_classifier()
dense = DenseRagRetriever()   # existing 'chunk and pray' baseline

default_out = {}
for cid in demo_ids:
    msg = cases[cid]['message']
    label, conf = clf.classify(msg)
    hit = dense.search(msg)[0]
    default_out[cid] = {'class': label, 'draft': hit['answer'], 'verified': hit['verified']}
    print(f"[{cid}] class={label} verified={hit['verified']}")
    print('   draft:', hit['answer'][:200])
    print()

## The governed system: it grounds, escalates or refuses

The same three messages through `build_complaint_harness`. The routine fee case runs the full workflow to a grounded draft; the "unfair" case raises a UDAAP flag and escalates instead of drafting; the SSN case is refused at the PII gate on the first tool call and escalates. Each outcome is a decision in the audit chain, not a silent draft.

In [ ]:
from agentlab.capstone import build_complaint_harness
from agentlab.core import Budget, BudgetTracker, TaskSpec

harness, registry = build_complaint_harness(policies_dir=root / 'data' / 'policies')

gov_out = {}
for cid in demo_ids:
    task = TaskSpec(goal='handle complaint', inputs={'message': cases[cid]['message']})
    traj = harness.run(task, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))
    out = traj.final_state.final_output or {}
    esc = next((r for r in traj.records if r.action.kind == 'escalate'), None)
    gov_out[cid] = {'status': traj.final_state.status,
                    'action': out.get('recommended_action'),
                    'reason': esc.action.reason if esc else '',
                    'draft': (out.get('draft_response') or '')}
    print(f"[{cid}] status={traj.final_state.status} action={out.get('recommended_action')}")
    if esc:
        print('   escalation:', esc.action.reason)
    elif out.get('draft_response'):
        print('   grounded draft:', out['draft_response'][:200])
    print()

## Side by side

The default drafts confidently on all three, unverified. The governed system grounds the one it should answer and stops on the two it should not.

In [ ]:
print(f"{'case':10s} {'default':28s} {'governed':28s}")
print('-' * 68)
for cid in demo_ids:
    d = f"drafts (verified={default_out[cid]['verified']})"
    g = gov_out[cid]['status']
    if gov_out[cid]['reason']:
        g += ': ' + gov_out[cid]['reason'][:40]
    print(f'{cid:10s} {d:28s} {g:28s}')

The task is identical. The difference is that every step of the governed run is bounded, grounded and checked, so a case it cannot safely answer becomes an escalation a human sees rather than a confident reply no one flagged.